# FreeVideo-Kaggle — Hindi Script to Video on T4/P100

Production low-VRAM AI video pipeline. Primary engine: **Wan2.1-T2V-1.3B (FP16)**.
FreeVideo H3 is included as an optional experimental engine (see AUDIT.md).

**Steps:** 1) Install 2) Probe kernels 3) Launch web UI + public link 4) Paste script → Generate → Download. Cell 6 = generate without the browser UI.
Settings: Accelerator = GPU T4 x2 (or P100), Internet = ON, Save version = Save.

In [ ]:
# Cell 1 — Install system + Python deps (CUDA 12.4 stack for T4/P100)
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1; echo ffmpeg done
!pip install -q torch==2.5.1+cu124 torchvision==0.20.1+cu124 --index-url https://download.pytorch.org/whl/cu124
!pip install -q diffusers==0.32.2 transformers==4.46.3 accelerate==1.1.1 safetensors==0.4.5 huggingface_hub==0.26.2 einops==0.8.0 sentencepiece==0.2.0 parler-tts==0.2.1 soundfile==0.12.1 "imageio[ffmpeg]==2.37.0" flask psutil numpy pillow requests python-dotenv av
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print(torch.cuda.get_device_capability(0) if torch.cuda.is_available() else 'n/a')

In [ ]:
# Cell 2 — Get the project + upstream FreeVideo
# >>> SET THIS to your own GitHub repo (the one you pushed freevideo-kaggle to) <<<
PROJECT_REPO = 'https://github.com/YOUR_USER/freevideo-kaggle.git'
# Alternative: upload the project as a Kaggle Dataset (zip or folder) and leave PROJECT_REPO as-is;
# the cell auto-detects any /kaggle/input/*/ containing app/server.py or a freevideo-kaggle*.zip.
import os, sys, glob, shutil, subprocess, zipfile
WORK = '/kaggle/working'; PROJ = f'{WORK}/freevideo-kaggle'; UP = f'{WORK}/FreeVideo'

def _have(p): return os.path.isfile(os.path.join(p, 'app', 'server.py'))

if not _have(PROJ):
    shutil.rmtree(PROJ, ignore_errors=True)
    if 'YOUR_USER' not in PROJECT_REPO:
        print('Cloning project from', PROJECT_REPO)
        subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO, PROJ], check=False)
    if not _have(PROJ):
        for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
                hit = [n for n in names if n.endswith('app/server.py')]
                if hit:
                    print('Unzipping dataset', z); zf.extractall(f'{WORK}/_unz')
                    src = os.path.join(f'{WORK}/_unz', hit[0][:-len('app/server.py')])
                    shutil.copytree(src, PROJ, dirs_exist_ok=True); break
    if not _have(PROJ):
        for d in glob.glob('/kaggle/input/**/app/server.py', recursive=True):
            src = os.path.dirname(os.path.dirname(d))
            print('Copying dataset folder', src); shutil.copytree(src, PROJ, dirs_exist_ok=True); break
if not _have(PROJ):
    raise SystemExit('PROJECT NOT FOUND. Either (a) push freevideo-kaggle to GitHub and set PROJECT_REPO above, '
                     'or (b) Add Input -> Upload -> New Dataset with the freevideo-kaggle folder/zip, then re-run this cell.')

if not os.path.isdir(f'{UP}/freevideo_engine'):
    shutil.rmtree(UP, ignore_errors=True)
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/FlashML-org/FreeVideo.git', UP], check=True)
sys.path.insert(0, PROJ); os.chdir(PROJ)
!python patches/apply_patches.py /kaggle/working/FreeVideo
print('project ready at', PROJ)

In [ ]:
# Cell 3 — Hardware + kernel probes (run BEFORE downloading any model)
import sys; sys.path.insert(0, '/kaggle/working/freevideo-kaggle')
from app.hardware import HardwareDetector
print(HardwareDetector().detect().to_dict())
!python scripts/probe_kernels.py --out /kaggle/working/kernel_probe.json
!python scripts/benchmark.py --quick --out /kaggle/working/benchmark_quick.json

In [ ]:
# Cell 4 — Persistent HF cache (survives sessions via /kaggle/working)
import os
os.environ['HF_HOME'] = '/kaggle/working/.hf_cache'
os.environ['HF_HUB_CACHE'] = '/kaggle/working/.hf_cache/hub'
os.environ['FREEVVIDEO_REPO'] = '/kaggle/working/FreeVideo'
print('HF cache ->', os.environ['HF_HOME'])

In [ ]:
# Cell 5 — Launch web UI + PUBLIC link (Kaggle has no port preview; we use a free Cloudflare quick tunnel)
import sys, threading, time, urllib.request
sys.path.insert(0, '/kaggle/working/freevideo-kaggle')
from app.server import run
from scripts.tunnel import start_tunnel
if not globals().get('_ui_thread'):
    _ui_thread = threading.Thread(target=run, kwargs={'host': '0.0.0.0', 'port': 5000}, daemon=True)
    _ui_thread.start(); time.sleep(2)
url = start_tunnel(5000)
for _ in range(20):  # wait for the tunnel DNS to go live (~10-20 s)
    try:
        urllib.request.urlopen(url + '/api/health', timeout=10); break
    except Exception:
        time.sleep(3)
print('\n' + '=' * 70)
print('OPEN THIS IN YOUR BROWSER  ->', url)
print('=' * 70)
print('Link stays alive only while this notebook session is running.')
print('Paste your Hindi script, preview voice, then Start generation.')

In [ ]:
# Cell 6 — (OPTIONAL, no browser needed) Generate directly from the notebook
# Use this if the tunnel link is blocked for you. Paste your script between the triple quotes.
SCRIPT = """
यहाँ अपनी हिंदी स्क्रिप्ट पेस्ट करें।
"""
ENGINE, PROFILE, STYLE, TTS = 'wan2.1', 'balanced', 'horror', 'indic-parler'
RESUME_PROJECT_ID = None   # set to a previous project_id to resume after a session cut

import sys; sys.path.insert(0, '/kaggle/working/freevideo-kaggle')
from app.pipeline import Pipeline
def _progress(stage, done, total, msg=''): print(f'[{stage}] {done}/{total} {msg}', flush=True)
if SCRIPT.strip() and 'अपनी हिंदी स्क्रिप्ट पेस्ट' not in SCRIPT:
    manifest = Pipeline(progress=_progress).run(SCRIPT.strip(), engine_id=ENGINE, profile=PROFILE,
                                               style=STYLE, tts_backend=TTS, resume_project_id=RESUME_PROJECT_ID)
    print('project_id:', manifest['project_id']); print('FINAL MP4:', manifest['final'])
    print('seconds:', manifest['seconds'], '| scenes:', len(manifest['scenes']))
    from IPython.display import Video, display; display(Video(manifest['final'], embed=True, width=640))
else:
    print('Paste your script into SCRIPT above, then re-run this cell.')

## Manual steps
1. Open the public `https://....trycloudflare.com` link printed by Cell 5 (Kaggle has no built-in port preview).
2. Paste your Hindi/Hinglish script (stored verbatim).
3. Select engine (Wan2.1 recommended), profile, style, TTS.
4. Preview narration audio.
5. Start generation → watch scene progress → preview scenes.
6. Download final MP4 (also saved to /kaggle/working/outputs/).
7. Optional: set TELEGRAM_BOT_TOKEN / TELEGRAM_CHAT_ID env vars for Telegram; YOUTUBE_CLIENT_SECRETS for YouTube upload.

## Session recovery
If the session ends mid-generation, re-run Cells 1–5, then either call the API with `resume_project_id` or set `RESUME_PROJECT_ID` in Cell 6 — completed scenes are skipped.